# S6E10 — 06 RealMLP (pytabkit) with the community settings

**Why:** our own PyTorch MLP (notebook 04) plateaued at OOF 0.96043. A public notebook ("Flight Log V3", built on Vladimir Demidov's RealMLP notebook as adapted by goodpjw2008) reports **0.96123 OOF for RealMLP** — better than its own LightGBM (0.96112) — with a tuned recipe. We run the same recipe from the `pytabkit` library on **our** feature set, for two reasons: (1) a strong, different ensemble member, (2) a benchmark of what a neural network can reach on this data, so we can see which ingredients our own MLP is missing.

**Recipe (RealMLP-TD, community settings):**
* route (`flight_distance`), age and every rating are given **twice**: as numbers and as categories ("categorical twins"); categories with ≤ 18 levels are one-hot encoded, the route (3.5k levels) gets a 5-dimensional embedding;
* numeric features pass through **PLR embeddings** (periodic + linear + ReLU: each feature is expanded by learnable sin/cos frequencies), inputs are robust-scaled, smoothly clipped and L2-normalised;
* very high learning rate 0.053 with a learning-rate schedule (`lin_cos_log_15`), the first layer at 0.25x of it, weight decay 0.0236 with its own schedule, `sq_mom` 0.988 (Adam β2), dropout 0.05 with a schedule, label smoothing 0.01;
* layers 512-256-128, SiLU, batch 256, **only 4 epochs**, **8 networks trained side by side** (n_ens=8, predictions averaged).

**Difference to the source notebook:** their feature set differs from ours (they add digits/GPT-2-token/survey-rate features; we have the route profile `fdm_*`). We use our `X_final` without the in-fold target encoding (the route embedding replaces it) and without `orig_prob` (we keep `orig_logit`).

**Plan:** (1) a timing probe to see how long one fold takes on this machine; (2) fold 0 as a benchmark (ours: 0.96018 for the own MLP, 0.96095 for tuned LightGBM); (3) all 5 folds, resumable, producing `preds/oof_realmlp.npy` / `test_realmlp.npy` for the stack.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import sys, subprocess, importlib.util
if importlib.util.find_spec('pytabkit') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit'], check=False)
import torch
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

SMOKE = False        # True: 60k training rows, 1 epoch, 2 networks: only checks that the code runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'      # 'mps' is not used (unstable in this environment)
N_ENS = 8 if DEVICE == 'cuda' else 4     # networks per model; 8 in the source notebook, 4 on CPU to keep the time acceptable
N_EPOCHS = 4
SEED = 2            # run the notebook again with SEED = 1, 2, ... to get more seeds (seed 0 is already done); they are averaged in section 6
print('device:', DEVICE, '| networks per model:', N_ENS, '| epochs:', N_EPOCHS)

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'X_final_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds')); PREDS_DIR.mkdir(exist_ok=True)
X = pd.read_parquet(PROC_DIR / 'X_final_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_final_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
print(X.shape, X_test.shape)

/opt/anaconda3/envs/kaggle-playground/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


device: cpu | networks per model: 4 | epochs: 4
(699635, 45) (299844, 45)


## 1. Input frame
`nn_frame` builds the RealMLP input: our columns, `orig_prob` dropped, `orig_logit` kept, and categorical twins for route, age and the 13 ratings. The category lists are taken from train + test together (fixed), so a fold never sees an unknown level.

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
FULL = pd.concat([X, X_test], ignore_index=True)
TWIN_LEVELS = {t: sorted(FULL[s].astype(int).unique()) for t, s in TWINS.items()}
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)

def nn_frame(D):
    D = D.drop(columns=['orig_prob']).copy()
    for t, s in TWINS.items():
        D[t] = pd.Categorical(D[s].astype(int), categories=TWIN_LEVELS[t])
    assert not D.isna().any().any()
    return D

print('categorical columns:', len(CAT_COLS), '| route levels:', len(TWIN_LEVELS['route_id']), '| input columns:', nn_frame(X.iloc[:5]).shape[1])

categorical columns: 19 | route levels: 3521 | input columns: 59


## 2. Parameters (the community recipe)

In [3]:
PARAMS = dict(
    n_epochs=1 if SMOKE else N_EPOCHS, n_ens=2 if SMOKE else N_ENS, batch_size=256, use_early_stopping=False,
    val_metric_name='1-auc_ovr', device=DEVICE, random_state=0, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False,
    bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)

def fit_predict(tr_idx, va_idx, seed=None, rows=None):
    seed = SEED if seed is None else seed
    tr_idx = tr_idx if rows is None else tr_idx[:rows]
    Xtr, Xva, Xte = nn_frame(X.iloc[tr_idx]), nn_frame(X.iloc[va_idx]), nn_frame(X_test)
    model = RealMLP_TD_Classifier(**{**PARAMS, 'random_state': seed})
    model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=CAT_COLS)
    p_va, p_te = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return p_va, p_te

## 3. Timing probe
One epoch-equivalent on 50,000 rows, to extrapolate the time of a full fold (≈ 560,000 rows x 4 epochs). Decide from this where to run the rest (this machine, the home GPU, Kaggle GPU).

In [4]:
tr0, va0 = FOLDS[0]
t0 = time.time()
old = PARAMS['n_epochs']; PARAMS['n_epochs'] = 1
_ = fit_predict(tr0, va0[:20000], rows=50000)
PARAMS['n_epochs'] = old
t_probe = time.time() - t0
est = t_probe * (len(tr0) / 50000) * (old if not SMOKE else 1)
print(f'probe: {t_probe:.0f} s for 50k rows x 1 epoch (includes setup and prediction of 20k + 300k test rows)')
print(f'rough estimate for one full fold: {est/60:.0f} min -> 5 folds: {5*est/3600:.1f} h  (an upper bound: setup and test prediction are included in the probe)')

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=1` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


probe: 14 s for 50k rows x 1 epoch (includes setup and prediction of 20k + 300k test rows)
rough estimate for one full fold: 11 min -> 5 folds: 0.9 h  (an upper bound: setup and test prediction are included in the probe)


## 4. Fold 0 — benchmark
Compare with the own MLP (fold 0: **0.96018**) and tuned LightGBM (fold 0: **0.96095**). The source notebook reports 0.96114 (their features) for RealMLP on fold 0.

In [5]:
t0 = time.time()
p_va0, p_te0 = fit_predict(*FOLDS[0], rows=60000 if SMOKE else None)
auc0 = roc_auc_score(y[FOLDS[0][1]], p_va0)
print(f'fold 0: RealMLP AUC = {auc0:.5f}  ({(time.time()-t0)/60:.1f} min)   | own MLP 0.96018 | tuned LightGBM 0.96095')

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 0: RealMLP AUC = 0.96087  (3.1 min)   | own MLP 0.96018 | tuned LightGBM 0.96095


## 5. All 5 folds (resumable)
Each finished fold is saved in `preds/realmlp_folds/`. Run this cell after reading the benchmark above; if the run is interrupted, running it again continues from the last finished fold.
Set `RUN_ALL = False` to skip it (e.g. while only the benchmark matters).

In [6]:
RUN_ALL = True
if RUN_ALL and not SMOKE:
    CK = PREDS_DIR / ('realmlp_folds' if SEED == 0 else f'realmlp_folds_s{SEED}'); CK.mkdir(exist_ok=True)
    oof = np.full(len(X), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        fo, ft = CK / f'oof_{k}.npy', CK / f'test_{k}.npy'
        if fo.exists() and ft.exists():
            oof[va] = np.load(fo); tests.append(np.load(ft)); print(f'fold {k}: loaded'); continue
        if k == 0:                                   # already computed above
            p_va, p_te = p_va0, p_te0
        else:
            t0 = time.time(); p_va, p_te = fit_predict(tr, va)
        oof[va] = p_va; tests.append(p_te); np.save(fo, p_va); np.save(ft, p_te)
        print(f'fold {k}: AUC {roc_auc_score(y[va], p_va):.5f}', flush=True)
    print(f'\nOOF AUC (RealMLP): {roc_auc_score(y, oof):.5f}')
    sfx = '' if SEED == 0 else f'_s{SEED}'
    np.save(PREDS_DIR / f'oof_realmlp{sfx}.npy', oof); np.save(PREDS_DIR / f'test_realmlp{sfx}.npy', np.mean(tests, axis=0))
    print('saved', PREDS_DIR / f'oof_realmlp{sfx}.npy', '(probabilities)')

fold 0: AUC 0.96087


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 1: AUC 0.95989


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 2: AUC 0.96163


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 3: AUC 0.96079


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 4: AUC 0.96062

OOF AUC (RealMLP): 0.96075
saved preds/oof_realmlp_s2.npy (probabilities)


## 6. Average the seeds
Averages the logits of all seeds found in `preds/` (seed 0 = `oof_realmlp.npy`, further seeds = `oof_realmlp_s1.npy`, ...) and writes `oof_realmlp_avg.npy` / `test_realmlp_avg.npy`. `05_stack` uses the averaged files automatically when they exist.

In [7]:
if not SMOKE:
    lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    sig = lambda z: 1 / (1 + np.exp(-z))
    seeds = [(0, PREDS_DIR / 'oof_realmlp.npy', PREDS_DIR / 'test_realmlp.npy')] + [(s, PREDS_DIR / f'oof_realmlp_s{s}.npy', PREDS_DIR / f'test_realmlp_s{s}.npy') for s in range(1, 10)]
    seeds = [t for t in seeds if t[1].exists() and t[2].exists()]
    O_ = [lg(np.load(o)) for _, o, _ in seeds]; T_ = [lg(np.load(t)) for _, _, t in seeds]
    for (s, _, _), o in zip(seeds, O_): print(f'seed {s}: OOF AUC {roc_auc_score(y, o):.5f}')
    o_avg, t_avg = np.mean(O_, axis=0), np.mean(T_, axis=0)
    print(f'average of {len(seeds)} seeds: OOF AUC {roc_auc_score(y, o_avg):.5f}')
    if len(seeds) > 1:
        np.save(PREDS_DIR / 'oof_realmlp_avg.npy', sig(o_avg)); np.save(PREDS_DIR / 'test_realmlp_avg.npy', sig(t_avg)); print('saved the averaged files')

seed 0: OOF AUC 0.96075
seed 1: OOF AUC 0.96071
seed 2: OOF AUC 0.96075
average of 3 seeds: OOF AUC 0.96082
saved the averaged files


## 7. Next
`05_stack.ipynb` already has `realmlp` (it takes the averaged files when they exist). Rerun it.